# BIKV v16 G2 RULER VT 4k comparison/rescue

This is one independent frozen shard. It can run in Colab or Kaggle and does not share a run id with any other shard. Completed rows are mirrored immediately to Google Drive in Colab. In Kaggle, run with Save Version / Run All; checkpoints and the rolling artifact zip are written to /kaggle/working.


In [ ]:
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys, zipfile
import torch
IS_KAGGLE = Path('/kaggle/input').exists() and Path('/kaggle/working').exists()
IS_COLAB = Path('/var/colab/hostname').exists() and not IS_KAGGLE
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('Platform:', 'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'local')
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 2**30, 1))


In [ ]:
WORK = Path('/content/bikv_v16_ruler_4k_work') if IS_COLAB else Path('/kaggle/working/bikv_v16_ruler_4k_work')
if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    MIRROR = Path('/content/drive/MyDrive/bikv_runs')
    candidates = list(Path('/content').glob('boundary_interaction_kv_v16_continuation_suite*.zip'))
    if not candidates:
        files.upload()
        candidates = list(Path('/content').glob('boundary_interaction_kv_v16_continuation_suite*.zip'))
elif IS_KAGGLE:
    MIRROR = None
    candidates = list(Path('/kaggle/input').rglob('boundary_interaction_kv_v16_continuation_suite*.zip'))
    candidates += list(Path('/kaggle/working').glob('boundary_interaction_kv_v16_continuation_suite*.zip'))
else:
    MIRROR = None
    candidates = list(Path.cwd().glob('boundary_interaction_kv_v16_continuation_suite*.zip'))
SOURCE = WORK / 'source_v16'
if SOURCE.exists():
    shutil.rmtree(SOURCE)
SOURCE.mkdir(parents=True, exist_ok=True)
if candidates:
    archive_path = max(candidates, key=lambda path: path.stat().st_mtime)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(SOURCE)
elif IS_KAGGLE:
    input_projects = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml') if (p.parent / 'src' / 'bikv').is_dir()]
    if len(input_projects) != 1:
        raise FileNotFoundError(f'Expected one extracted BIKV Dataset project, found {input_projects}')
    shutil.copytree(input_projects[0], SOURCE / 'boundary_interaction_kv', dirs_exist_ok=True)
else:
    raise FileNotFoundError('Provide boundary_interaction_kv_v16_continuation_suite.zip.')
projects = list(SOURCE.rglob('pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError(f'Expected one project, found {len(projects)}')
PROJECT = projects[0].parent
print('Platform:', 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local', '| Project:', PROJECT)


In [ ]:
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'nltk>=3.9,<4'], check=True)
if sys.version_info >= (3, 13):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'fire==0.7.1'], check=True)
env = {**os.environ, 'PYTHONPATH': f"{PROJECT / 'src'}:{PROJECT / 'scripts'}"}
def stream(command):
    print(' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
print('Installed frozen BIKV continuation suite.')


## Frozen official benchmark

Uses the pinned NVIDIA RULER Variable Tracking generator at revision c3f5e3b and 50 examples. At 8k the failed full-context gate remains visible; compressed methods run as an exploratory rescue test. The method list and frozen selector weights are unchanged.


In [ ]:
LENGTH = 4096
RULER_DATA = (Path('/content/drive/MyDrive/bikv_datasets/v16_ruler_vt_4k_seed_20261016') if IS_COLAB else WORK / 'ruler_data')
RULER = WORK / 'RULER'
if not RULER.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/NVIDIA/RULER.git', str(RULER)], check=True)
subprocess.run(['git', '-C', str(RULER), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(RULER), 'checkout', '--detach', 'c3f5e3b'], check=True)
subprocess.run([sys.executable, str(PROJECT / 'scripts' / 'prepare_ruler_vt.py'), '--repo', str(RULER), '--output', str(RULER_DATA), '--model', 'Qwen/Qwen2.5-3B-Instruct', '--revision', 'c3f5e3b', '--lengths', str(LENGTH), '--samples', '50', '--seed', '20261016'], cwd=PROJECT, env=env, check=True)
print((RULER_DATA / 'ruler_manifest.json').read_text())


In [ ]:
base = json.loads((PROJECT / 'configs' / 'v15_g2_ruler_qualification.json').read_text())
base['run_id'] = 'bikv-v16-g2-ruler-vt-4k-comparison-rescue-seed-20261016'
dataset = RULER_DATA / f'vt_{LENGTH}' / 'validation.jsonl'
base['data_path'] = str(dataset)
base['ruler_manifest_path'] = str(RULER_DATA / 'ruler_manifest.json')
base['ruler_datasets'] = {str(LENGTH): str(dataset)}
base['frozen_models_path'] = str(PROJECT / 'configs' / 'v14_g1b_g2_frozen_models.json')
base['ruler_run_failed_gate_as_rescue'] = True
base['ruler_qualification_only'] = False
base['run_g1b'] = False
base['run_g2'] = True
base['output_root'] = str(WORK / 'runs')
base['mirror_root'] = str(MIRROR) if MIRROR else None
runtime_path = WORK / f'runtime_ruler_{LENGTH}.json'
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(base, indent=2), encoding='utf-8')
print(runtime_path.read_text())


## Run or resume

Re-run this cell after a Colab disconnect. Existing rows and captures are restored from Drive and skipped. A time-budget pause is normal: reconnect and run the same cell again.


In [ ]:
stream([sys.executable, '-u', str(PROJECT / 'scripts' / 'stage_g1b_g2.py'), '--config', str(runtime_path)])


In [ ]:
run_root = Path(base['output_root']) / base['run_id']
stream([sys.executable, str(PROJECT / 'scripts' / 'analyze_stage_g1b_g2.py'), str(run_root), '--seed', str(base['seed'])])
from bikv.storage import ResultStore
store = ResultStore(base['output_root'], base['run_id'], base.get('mirror_root'))
bundle = store.bundle()
print('STATUS:', (run_root / 'status.json').read_text())
print('REPORT:', (run_root / 'analysis' / 'report.md').read_text())
print('Artifact:', bundle)
if IS_COLAB:
    from google.colab import files
    files.download(str(bundle))
else:
    print('Kaggle output is in /kaggle/working. Save the notebook version to retain it.')
